# DLVT — Dynamic Leadership Vitality Theory
## Interactive Exploration Notebook

This notebook provides a guided tour of the DLVT model:

1. **Core ODE system** — vitality and capital dynamics
2. **Three outcome regimes** — sustainable, zombie, collapse
3. **Phase portrait analysis** — nullclines and equilibria
4. **Parameter sensitivity** — how β and R shape the carrying capacity
5. **Regime map** — which parameter combinations lead to which outcome

> **Reference:** Bendinelli, W. (2026). *The Carrying Capacity of Leadership: A Dynamic Theory of Executive Sustainability Under Endogenous Complexity Growth.* arXiv preprint.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from dlvt.model    import make_params, complexity, simulate, DEFAULT_PARAMS
from dlvt.analysis import (carrying_capacity, find_interior_equilibria,
                            jacobian_eigenvalues, is_zombie, classify_regime)

print('DLVT loaded. Default parameters:')
for k, v in DEFAULT_PARAMS.items():
    print(f'  {k:8s} = {v}')

---
## 1. The Core ODE System

The DLVT model tracks two state variables:

$$\frac{dV}{dt} = R\left(1 - \frac{V}{V_{\max}}\right) - \delta\, O^\gamma \cdot \frac{V}{V + \varepsilon}$$

$$\frac{dC}{dt} = \alpha\, I - \mu C, \quad I = \frac{C \cdot V}{1 + \phi O}, \quad O = O_0 + \beta C^\eta$$

The **depletion ratio** $\Gamma(t) = \delta O^\gamma / R$ tells us whether the system is in net drain ($\Gamma > 1$) or net recovery ($\Gamma < 1$).

In [ ]:
# Simulate baseline trajectory
p = make_params(beta=0.25)
t, V, C, O, I, G = simulate(p, V0=8.0, C0=0.5, T=150)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True,
                              gridspec_kw={'height_ratios': [2.5, 1]})
a1.plot(t, V, 'b-',  lw=2.5, label='V(t) — Vitality')
a1.plot(t, C, 'r--', lw=2,   label='C(t) — Career Capital')
a1.plot(t, O, color='darkorange', ls=':', lw=2, label='O(t) — Complexity')
a1.axhline(0.5*p['Vmax'], color='purple', ls='-.', alpha=0.5, label='V_strategic')
a1.legend(); a1.set_ylabel('Level'); a1.grid(alpha=0.2)
a1.set_title('DLVT Baseline Trajectory (β=0.25 → Zombie Equilibrium)')

a2.plot(t, G, 'k-', lw=2)
a2.axhline(1.0, color='red', ls='--', alpha=0.7, label='Γ=1 (critical threshold)')
a2.fill_between(t, 1, G, where=(G>1), alpha=0.15, color='red', label='Net drain')
a2.fill_between(t, G, 1, where=(G<1), alpha=0.15, color='green', label='Net recovery')
a2.legend(fontsize=9); a2.set_xlabel('Time'); a2.set_ylabel('Γ(t)'); a2.grid(alpha=0.2)
plt.tight_layout()

---
## 2. Equilibrium Analysis

At equilibrium: $dV/dt = dC/dt = 0$. The dC/dt = 0 nullcline gives:

$$V^* = \frac{\mu(1 + \phi O^*)}{\alpha}$$

Substituting into dV/dt = 0 gives a scalar equation in $C^*$ (solved numerically).

In [ ]:
# Find and classify equilibria
p = make_params(beta=0.25)
eqs = find_interior_equilibria(p)
cc  = carrying_capacity(p)

print(f'Carrying capacity C*_max = {cc:.3f}')
print(f'V_strategic = {0.5*p["Vmax"]:.1f}\n')
print(f'Found {len(eqs)} equilibrium/ia:')
for eq in eqs:
    regime = 'ZOMBIE' if eq['zombie'] else 'SUSTAINABLE'
    print(f'  C* = {eq["C"]:.3f},  V* = {eq["V"]:.3f},  I* = {eq["I"]:.3f}')
    print(f'  Stable: {eq["stable"]},  Regime: {regime}')
    print(f'  Eigenvalues: λ₁={eq["eigenvalues"][0]:.4f}, λ₂={eq["eigenvalues"][1]:.4f}')

---
## 3. Explore: How β (coupling) changes the regime

In [ ]:
# Try different β values and see how the equilibrium shifts
betas_to_try = [0.05, 0.10, 0.20, 0.35, 0.60]
fig, axes = plt.subplots(1, len(betas_to_try), figsize=(16, 4), sharey=True)

for ax, b in zip(axes, betas_to_try):
    p = make_params(beta=b)
    t, V, C, *_ = simulate(p, T=200)
    eqs = find_interior_equilibria(p)
    regime = classify_regime(p)
    
    color = {'sustainable': 'green', 'zombie': 'orange', 'collapse-prone': 'red'}[regime]
    ax.plot(t, V, color=color, lw=2, label='V(t)')
    ax.axhline(0.5*p['Vmax'], color='purple', ls='-.', alpha=0.4)
    ax.set_title(f'β={b}\n{regime}', fontsize=10)
    ax.set_xlabel('Time')
    if eqs:
        ax.axhline(eqs[0]['V'], color=color, ls=':', alpha=0.6)
    ax.grid(alpha=0.2)

axes[0].set_ylabel('Vitality V(t)')
plt.suptitle('Effect of capital-complexity coupling β on leadership outcome', y=1.02)
plt.tight_layout()

---
## 4. Carrying Capacity Sensitivity

$$C^*_{\max} = \left(\frac{(R/\delta)^{1/\gamma} - O_0}{\beta}\right)^{1/\eta}$$

Key insight: C*_max is **decreasing in β** and **increasing in R** — meaning tighter coupling depletes capacity faster, while better recovery expands it.

In [ ]:
betas = np.linspace(0.05, 1.0, 150)
Rs    = np.linspace(0.5, 6.0, 150)
Z = np.zeros((len(Rs), len(betas)))
for i, Rv in enumerate(Rs):
    for j, bv in enumerate(betas):
        Z[i,j] = carrying_capacity(make_params(R=Rv, beta=bv))

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.pcolormesh(betas, Rs, Z, cmap='RdYlGn', shading='auto')
plt.colorbar(im, ax=ax, label=r'C*_max')
CS = ax.contour(betas, Rs, Z, levels=[5,10,20,40], colors='black', lw=0.8, alpha=0.6)
ax.clabel(CS, inline=True, fontsize=8)
ax.set_xlabel('β (capital-complexity coupling)')
ax.set_ylabel('R (recovery rate)')
ax.set_title('Leadership Carrying Capacity C*(β, R)')
plt.tight_layout()

---
## 5. Custom Simulation

Explore your own parameter combinations.

In [ ]:
# === EDIT THESE PARAMETERS ===
custom_params = dict(
    R     = 4.0,    # Higher R = better recovery infrastructure
    beta  = 0.15,   # Lower β = better role design (less coupling)
    delta = 0.015,  # Lower δ = less energetically costly complexity
)
# ==============================

p = make_params(**custom_params)
t, V, C, O, I, G = simulate(p, V0=8.0, C0=0.5, T=150)

eqs    = find_interior_equilibria(p)
cc     = carrying_capacity(p)
regime = classify_regime(p)

print(f'Regime: {regime.upper()}')
print(f'Carrying capacity C*_max = {cc:.2f}')
if eqs:
    print(f'Equilibrium: C* = {eqs[0]["C"]:.2f}, V* = {eqs[0]["V"]:.2f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t, V, 'b-', lw=2, label='V(t)')
ax.plot(t, C, 'r--', lw=2, label='C(t)')
ax.axhline(0.5*p['Vmax'], color='purple', ls='-.', alpha=0.5, label='V_strategic')
ax.set_title(f'Custom simulation — Regime: {regime}  |  β={custom_params.get("beta", DEFAULT_PARAMS["beta"])}, R={custom_params.get("R", DEFAULT_PARAMS["R"])}')
ax.legend(); ax.set_xlabel('Time'); ax.set_ylabel('Level'); ax.grid(alpha=0.2)
plt.tight_layout()

---
## 6. Reproduce All Paper Figures

Running the cell below regenerates all 7 publication figures and saves them to `../figures/`.

In [ ]:
from dlvt.figures import generate_all
generate_all(output_dir='../figures/')